# Hafta 13 · Kuantum Makine Öğrenmesi II: Kuantum Çekirdekler ve QSVM
**Ders:** Kuantum Hesaplama ve Uygulamaları · **Lab süresi:** ~50 dk · **Ortam:** Google Colab (CPU yeterli)

12\. haftada veriyi kübitlere yazmayı (veri kodlama) ve iki kodlanmış durumun benzerliğini gördük. Bu hafta o benzerliği bir **çekirdek (kernel)** olarak kullanıp klasik SVM'e veriyoruz: **QSVM**. Sonuçları doğrusal ve RBF SVM ile dürüstçe karşılaştıracağız.

| Bölüm | Konu | Süre |
|---|---|---|
| 0 | Kurulum, veri setleri | 3 dk |
| A | Klasik SVM hatırlatma: Gram matrisi, `SVC(kernel="precomputed")` | 6 dk |
| B | Kuantum çekirdek: Statevector ile kesin hesap, açı kodlaması formülü | 7 dk |
| C | Devre ile çekirdek: compute–uncompute (shot) ve swap test | 7 dk |
| D | Özellik haritaları: açı, `zz_feature_map`, kendi haritamız; vektörleştirilmiş simülasyon | 5 dk |
| E | QSVM deneyleri: ısı haritaları, karar sınırları, karşılaştırma tablosu (5-kat CV) | 10 dk |
| F | Bant genişliği (γ) taraması | 4 dk |
| G | Shot tabanlı QSVM (küçük alt küme) ve maliyet hesabı | 5 dk |
| H | Çekirdek yoğunlaşması (exponential concentration) | 3 dk |
| I | PennyLane ile aynı çekirdek (genel kültür) | ödev |
| J | Alıştırmalar (8 adet, `assert` ile kontrol) | ödev |

## 0 · Kurulum

In [ ]:
!pip install -q qiskit qiskit-aer pylatexenc scikit-learn pennylane

In [ ]:
import time, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap, LinearSegmentedColormap
from qiskit import QuantumCircuit, transpile
from qiskit.circuit import ParameterVector
from qiskit.circuit.library import zz_feature_map
from qiskit.quantum_info import Statevector
from qiskit_aer import AerSimulator
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.metrics.pairwise import rbf_kernel

np.set_printoptions(precision=3, suppress=True)
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})
NAVY, BLUE, ORANGE, GRAY = "#1F3A5F", "#2E6DB4", "#D9822B", "#8A94A6"
CM2 = ListedColormap(["#D6E4F5", "#F8E3CC"])                                   # karar bölgeleri
CMK = LinearSegmentedColormap.from_list("k", ["#FFFFFF", "#9DBBE3", NAVY])      # çekirdek ısı haritası
SEED = 42
T_START = time.time()
print("hazır")

### Veri setleri
Aşağıdaki fonksiyonlar dersin ortak veri seti kodudur (her hafta aynı). Tüm özellikler **[0, π]** aralığına ölçeklenmiştir, yani doğrudan açı olarak kullanılabilir.
Bu veri setlerinin CSV'leri ayrıca verilmiştir: `moons.csv`, `circles.csv`, `xor.csv`, `iris_2sinif.csv`.

In [ ]:
from sklearn.datasets import make_moons, make_circles, load_iris
from sklearn.preprocessing import MinMaxScaler

def ds_moons(n=200, noise=0.15):
    X, y = make_moons(n_samples=n, noise=noise, random_state=SEED)
    X = MinMaxScaler((0, np.pi)).fit_transform(X)            # açı kodlaması için [0, π]
    return pd.DataFrame({"x1": X[:, 0], "x2": X[:, 1], "y": y})

def ds_circles(n=200, noise=0.08, factor=0.45):
    X, y = make_circles(n_samples=n, noise=noise, factor=factor, random_state=SEED)
    X = MinMaxScaler((0, np.pi)).fit_transform(X)
    return pd.DataFrame({"x1": X[:, 0], "x2": X[:, 1], "y": y})

def ds_xor(n=200, noise=0.12):
    rng = np.random.default_rng(SEED)
    c = rng.integers(0, 2, size=(n, 2))
    X = c + rng.normal(0, noise, size=(n, 2))
    y = (c[:, 0] ^ c[:, 1]).astype(int)
    X = MinMaxScaler((0, np.pi)).fit_transform(X)
    return pd.DataFrame({"x1": X[:, 0], "x2": X[:, 1], "y": y})

def ds_iris2():
    """Iris: versicolor (0) ve virginica (1) — doğrusal olarak tam ayrılamayan iki sınıf; 4 özellik."""
    d = load_iris()
    m = d.target > 0
    X = MinMaxScaler((0, np.pi)).fit_transform(d.data[m])
    y = d.target[m] - 1
    cols = ["sepal_len", "sepal_wid", "petal_len", "petal_wid"]
    df = pd.DataFrame(X, columns=cols); df["y"] = y
    return df


DS = {"moons": ds_moons(), "circles": ds_circles(), "xor": ds_xor(), "iris2": ds_iris2()}
SPLIT = {}
for k, df in DS.items():
    X = df.drop(columns="y").values; y = df.y.values
    SPLIT[k] = train_test_split(X, y, test_size=0.3, random_state=SEED, stratify=y) + [X, y]
    print(f"{k:8s} X={X.shape}  sınıf dağılımı={np.bincount(y)}")

fig, axs = plt.subplots(1, 4, figsize=(15, 3.6))
for ax, k in zip(axs, DS):
    Xa, ya = SPLIT[k][4], SPLIT[k][5]; i0, i1 = (2, 3) if k == "iris2" else (0, 1)
    ax.scatter(Xa[ya == 0, i0], Xa[ya == 0, i1], c=BLUE, s=12); ax.scatter(Xa[ya == 1, i0], Xa[ya == 1, i1], c=ORANGE, s=12)
    ax.set_title(k, color=NAVY); ax.set_aspect("equal")
plt.tight_layout(); plt.show()

---
## A · Klasik SVM hatırlatma
SVM iki sınıfı ayıran ve **marjini en geniş** olan sınırı arar. Eğitim sonunda sınırı yalnızca **destek vektörleri** belirler. Tahmin fonksiyonu:
$$f(x) = \text{sign}\Big(\sum_{i \in SV} \alpha_i y_i \, k(x_i, x) + b\Big)$$
Dikkat: veri yalnızca **k(x, x')** üzerinden girer. Bu, **çekirdek hilesidir**: özellik uzayını açıkça kurmadan sadece benzerlik fonksiyonunu vermek yeterlidir.

| Çekirdek | Formül | sklearn |
|---|---|---|
| Doğrusal | k = x·x' | `SVC(kernel="linear")` |
| RBF | k = exp(−γ‖x − x'‖²) | `SVC(kernel="rbf", gamma=...)` |
| Hazır matris | K[i, j] = k(xᵢ, xⱼ) kendimiz hesaplarız | `SVC(kernel="precomputed")` |

**Kuantum çekirdek için anahtar araç** üçüncü satırdır: çekirdek matrisini (Gram matrisi) kendimiz hesaplayıp SVM'e veririz.

In [ ]:
Xtr, Xte, ytr, yte, Xall, yall = SPLIT["moons"]

# 1) sklearn'ün kendi RBF'i
m1 = SVC(kernel="rbf", gamma=1.0).fit(Xtr, ytr)
# 2) Aynı çekirdeği biz hesaplayıp "precomputed" olarak verelim
K_train = rbf_kernel(Xtr, Xtr, gamma=1.0)          # (N_train, N_train)
K_test  = rbf_kernel(Xte, Xtr, gamma=1.0)          # (N_test, N_train)  <- satır: test, sütun: EĞİTİM
m2 = SVC(kernel="precomputed").fit(K_train, ytr)

print("rbf        test doğruluğu:", m1.score(Xte, yte))
print("precomputed test doğruluğu:", m2.score(K_test, yte))
print("tahminler aynı mı?", np.array_equal(m1.predict(Xte), m2.predict(K_test)))
print("destek vektörü sayısı:", len(m2.support_), "/", len(ytr))

In [ ]:
# Gram matrisinin görünümü: sınıfa göre sıralayınca blok yapı görünür
o = np.argsort(ytr, kind="stable")
fig, axs = plt.subplots(1, 2, figsize=(10, 4.2))
for ax, (tl, Kx) in zip(axs, [("Doğrusal  k = x·x'", Xtr[o] @ Xtr[o].T), ("RBF  k = exp(−‖x−x'‖²)", rbf_kernel(Xtr[o], gamma=1.0))]):
    im = ax.imshow(Kx, cmap=CMK); plt.colorbar(im, ax=ax, fraction=0.046); ax.set_title(tl, color=NAVY)
plt.tight_layout(); plt.show()

---
## B · Kuantum çekirdek
**Fikir:** Veriyi bir devreyle kübitlere yaz (özellik haritası, *feature map*): x → |φ(x)⟩ = U(x)|0…0⟩. İki durumun benzerliği:
$$k(x, x') = |\langle \varphi(x) | \varphi(x') \rangle|^2$$
- k(x, x) = 1, 0 ≤ k ≤ 1, simetrik → geçerli bir çekirdek (Gram matrisi pozitif yarı-tanımlı).
- 2\. haftadaki genel olasılık kuralıyla aynı ifade: "|φ(x')⟩ durumu, |φ(x)⟩ olarak bulunma olasılığı".

### (a) Statevector ile kesin hesap

In [ ]:
def angle_circuit(x):
    """Açı kodlaması: kübit i'ye RY(x_i)."""
    qc = QuantumCircuit(len(x))
    for i, xi in enumerate(x):
        qc.ry(xi, i)
    return qc

def kernel_statevector(x, xp, circuit_fn=angle_circuit):
    a = Statevector(circuit_fn(x)).data
    b = Statevector(circuit_fn(xp)).data
    return abs(np.vdot(a, b))**2

x, xp = np.array([0.5, 1.2]), np.array([1.5, 0.4])
print("Statevector   k =", kernel_statevector(x, xp))
print("Kapalı formül k = Π cos²((xᵢ − x'ᵢ)/2) =", np.prod(np.cos((x - xp)/2)**2))
angle_circuit(x).draw("mpl")

**Neden kapalı formül?** Tek kübitte RY(θ)|0⟩ = [cos(θ/2), sin(θ/2)]. İki durum için
⟨φ(x)|φ(x')⟩ = cos(x/2)cos(x'/2) + sin(x/2)sin(x'/2) = cos((x − x')/2). Kübitler bağımsız (çarpım durumu) olduğundan çok kübitte çarpılır:
$$k_{\text{açı}}(x, x') = \prod_i \cos^2\!\Big(\frac{x_i - x'_i}{2}\Big)$$
Bu çekirdek yalnızca **farka** bağlıdır — RBF'e çok benzer bir "yerel benzerlik" ölçüsüdür.

In [ ]:
d = np.linspace(-np.pi, np.pi, 300)
plt.figure(figsize=(7, 3.4))
for g, c in [(0.5, GRAY), (1.0, BLUE), (2.0, NAVY)]:
    plt.plot(d, np.cos(g*d/2)**2, color=c, label=f"açı çekirdeği, γ={g}")
plt.plot(d, np.exp(-d**2/2), ":", color=ORANGE, label="RBF exp(−Δ²/2)")
plt.xlabel("Δ = x − x'"); plt.ylabel("k"); plt.legend(frameon=False); plt.grid(alpha=.25); plt.show()

---
## C · Devre ile çekirdek: compute–uncompute ve swap test
Gerçek donanımda durum vektörünü **okuyamayız**, yalnızca ölçebiliriz. Çözüm:
$$k(x, x') = |\langle 0 | U(x')^\dagger U(x) | 0 \rangle|^2 = P(\text{tümü sıfır})$$
Devre: önce **U(x)** (compute), sonra **U(x')†** (uncompute), sonra ölç. x = x' ise U† her şeyi geri alır ve her shot `00…0` verir (k = 1).

In [ ]:
def cu_circuit(fmap, x, xp):
    """compute–uncompute devresi: U(x) ardından U(x')†, sonra ölçüm."""
    qc = fmap(x).compose(fmap(xp).inverse())
    qc.measure_all()
    return qc

qc = cu_circuit(angle_circuit, x, xp)
aer = AerSimulator(seed_simulator=SEED)
shots = 2000
counts = aer.run(transpile(qc, aer), shots=shots).result().get_counts()
print("sayımlar:", counts)
print("shot tahmini k ≈", counts.get("00", 0)/shots, "   kesin:", kernel_statevector(x, xp))
qc.draw("mpl")

### Swap test (kısa)
U† kullanmadan iki durumun örtüşmesini ölçmenin bir diğer yolu: bir yardımcı kübit + **CSWAP**. P(yardımcı = 0) = (1 + k)/2 → **k = 2·P(0) − 1**. Bedeli: 2n + 1 kübit ve pahalı CSWAP kapıları. Pratikte çekirdek hesabında genellikle compute–uncompute tercih edilir.

In [ ]:
a0, b0 = 0.5, 1.5                        # 1 kübitlik iki veri noktası
sw = QuantumCircuit(3, 1)
sw.h(0); sw.ry(a0, 1); sw.ry(b0, 2)
sw.cswap(0, 1, 2); sw.h(0); sw.measure(0, 0)
c = aer.run(transpile(sw, aer), shots=4000).result().get_counts()
p0 = c.get("0", 0)/4000
print(f"P(0) = {p0:.3f} -> k ≈ 2P(0) − 1 = {2*p0 - 1:.3f}   kesin: {np.cos((a0 - b0)/2)**2:.3f}")
sw.draw("mpl")

---
## D · Özellik haritaları ve hızlı (vektörleştirilmiş) simülasyon
| Harita | Devre | Özellik |
|---|---|---|
| Açı | RY(γxᵢ) her kübite | Çarpım durumu; çekirdek = Π cos²(γΔᵢ/2) |
| ZZ | [H → P(2γxᵢ) → CX–P(2(π−γxᵢ)(π−γxⱼ))–CX] × reps | Qiskit `zz_feature_map`; kübitler arası etkileşim terimi |
| Özel | [RY(γxᵢ) → CZ zinciri → RZ(γxᵢ)] × reps | Bizim tasarımımız |

**Qiskit 2.x notu:** `ZZFeatureMap` **sınıfı** Qiskit 2.1'den beri kullanımdan kaldırılma (deprecated) sürecinde; önerilen kullanım `zz_feature_map(n, reps)` **fonksiyonudur** ve düz bir `QuantumCircuit` döndürür.

**γ (bant genişliği):** Haritaya x yerine γ·x veririz. γ, çekirdeğin "ne kadar yakın noktalara benzer diyeceğini" ayarlar (RBF'teki γ gibi).

In [ ]:
fm = zz_feature_map(2, reps=2)
print("parametreler:", fm.parameters)
fm.decompose().draw("mpl", fold=-1)

Her (x, x') çifti için ayrı devre simüle etmek yavaştır (200 örnek → 20 000 devre). Küçük kübit sayısında durum vektörlerini **NumPy ile toplu** hesaplayıp tüm Gram matrisini tek matris çarpımıyla alabiliriz:
$$K = |S_A^{*} \, S_B^{T}|^2 \qquad (S: \text{her satırı bir örneğin durum vektörü})$$
Aşağıdaki fonksiyonlar Qiskit devreleriyle **birebir aynı** durumları üretir (hemen ardından test ediyoruz).

In [ ]:
import numpy as np
from scipy.linalg import hadamard


def bit_table(n):
    """(2^n, n) tablo: satır = indeks, sütun i = kübit i'nin biti (Qiskit sırası)."""
    idx = np.arange(2 ** n)
    return (idx[:, None] >> np.arange(n)[None, :]) & 1


def angle_states(X, gamma=1.0):
    """Açı kodlaması: her özellik kendi kübitine RY(γ·x_i) ile yazılır (çarpım durumu)."""
    X = gamma * np.atleast_2d(np.asarray(X, dtype=float))
    B, n = X.shape
    S = np.ones((B, 1), dtype=complex)
    for i in reversed(range(n)):                      # kron zinciri: q_{n-1} ⊗ ... ⊗ q_0
        v = np.stack([np.cos(X[:, i] / 2), np.sin(X[:, i] / 2)], axis=1)
        S = (S[:, :, None] * v[:, None, :]).reshape(B, -1)
    return S


def zz_states(X, gamma=1.0, reps=2):
    """Qiskit zz_feature_map(n, reps) ile birebir aynı durum: [H katmanı → köşegen faz] × reps.
    Faz(b) = Σ_i 2·x_i·b_i + Σ_{i<j} 2·(π − x_i)(π − x_j)·(b_i XOR b_j)."""
    X = gamma * np.atleast_2d(np.asarray(X, dtype=float))
    B, n = X.shape
    b = bit_table(n)
    phase = 2 * X @ b.T
    for i in range(n):
        for j in range(i + 1, n):
            phase += 2 * np.outer((np.pi - X[:, i]) * (np.pi - X[:, j]), b[:, i] ^ b[:, j])
    D = 2 ** n
    Hn = hadamard(D) / np.sqrt(D)                     # H ⊗ H ⊗ ... ⊗ H
    S = np.zeros((B, D), dtype=complex); S[:, 0] = 1
    for _ in range(reps):
        S = (S @ Hn) * np.exp(1j * phase)
    return S


def _apply_1q(S, U, q, n):
    """Her örneğe kendi 2x2 kapısını (U: (B,2,2)) q kübitinde uygular."""
    B = S.shape[0]
    T = S.reshape(B, 2 ** (n - 1 - q), 2, 2 ** q)
    return np.einsum("bij,bajc->baic", U, T).reshape(B, -1)


def custom_states(X, gamma=1.0, reps=2):
    """Bizim özel haritamız: [RY(γx_i) → CZ zinciri → RZ(γx_i)] × reps."""
    X = gamma * np.atleast_2d(np.asarray(X, dtype=float))
    B, n = X.shape
    b = bit_table(n)
    cz = np.ones(2 ** n)
    for i in range(n - 1):
        cz *= (-1.0) ** (b[:, i] & b[:, i + 1])
    S = np.zeros((B, 2 ** n), dtype=complex); S[:, 0] = 1
    for _ in range(reps):
        for i in range(n):
            c, s = np.cos(X[:, i] / 2), np.sin(X[:, i] / 2)
            S = _apply_1q(S, np.stack([np.stack([c, -s], 1), np.stack([s, c], 1)], 1), i, n)
        S = S * cz
        rz = np.exp(-0.5j * X @ (1 - 2 * b).T)        # Π_i RZ(x_i): faz e^{-i x_i/2} (bit 0) / e^{+i x_i/2} (bit 1)
        S = S * rz
    return S


FMAPS = {"angle": angle_states, "zz": zz_states, "custom": custom_states}


def qkernel(XA, XB=None, fmap="zz", gamma=1.0):
    """Kuantum çekirdek matrisi K[i, j] = |⟨φ(a_i)|φ(b_j)⟩|² (Statevector ile kesin)."""
    f = FMAPS[fmap] if isinstance(fmap, str) else fmap
    SA = f(XA, gamma=gamma)
    SB = SA if XB is None else f(XB, gamma=gamma)
    return np.abs(SA.conj() @ SB.T) ** 2


def shot_kernel(K, shots, seed=0):
    """Shot gürültülü çekirdek: her eleman Binom(shots, k)/shots (compute–uncompute devresinin istatistiği).
    Simetri korunur, köşegen 1 bırakılır (x ile x için devre çalıştırmaya gerek yoktur)."""
    rng = np.random.default_rng(seed)
    Kn = rng.binomial(shots, np.clip(K, 0, 1)) / shots
    if K.shape[0] == K.shape[1] and np.allclose(K, K.T):
        Kn = np.triu(Kn, 1); Kn = Kn + Kn.T; np.fill_diagonal(Kn, 1.0)
    return Kn

In [ ]:
# Test: vektörleştirilmiş durumlar == Qiskit Statevector (global faz hariç)
rng = np.random.default_rng(1)
for n in [2, 3, 4]:
    Xr = rng.uniform(0, np.pi, (3, n))
    S = zz_states(Xr, gamma=0.7)
    for k in range(3):
        sv = Statevector(zz_feature_map(n, reps=2).assign_parameters(0.7*Xr[k])).data
        assert np.isclose(abs(np.vdot(sv, S[k])), 1.0)
    S = custom_states(Xr, gamma=0.7)
    for k in range(3):
        qc = QuantumCircuit(n)
        for _ in range(2):
            for i in range(n): qc.ry(0.7*Xr[k, i], i)
            for i in range(n - 1): qc.cz(i, i + 1)
            for i in range(n): qc.rz(0.7*Xr[k, i], i)
        assert np.isclose(abs(np.vdot(Statevector(qc).data, S[k])), 1.0)
print("zz_states ve custom_states Qiskit ile birebir aynı ✓")
Xa = rng.uniform(0, np.pi, (5, 2))
print("açı çekirdeği = kapalı formül ?", np.allclose(qkernel(Xa, fmap="angle"), np.prod(np.cos((Xa[:, None] - Xa[None])/2)**2, axis=2)))

t = time.time(); K = qkernel(SPLIT["moons"][0], fmap="zz", gamma=0.5)
print(f"140×140 ZZ Gram matrisi {time.time()-t:.3f} s'de hesaplandı; simetrik: {np.allclose(K, K.T)}, köşegen=1: {np.allclose(np.diag(K), 1)}")
print("en küçük özdeğer (≥ 0 olmalı):", np.linalg.eigvalsh(K).min().round(8))

In [ ]:
# Kendi haritamızın devresi (3 kübit, reps=2)
xv = ParameterVector("x", 3); qc = QuantumCircuit(3)
for r in range(2):
    for i in range(3): qc.ry(xv[i], i)
    qc.cz(0, 1); qc.cz(1, 2)
    for i in range(3): qc.rz(xv[i], i)
    if r == 0: qc.barrier()
qc.draw("mpl", fold=-1)

---
## E · QSVM deneyleri
Akış: **Gram matrisi (eğitim×eğitim)** → `SVC(kernel="precomputed").fit` → **test×eğitim** çekirdeği → `score`. Karar sınırı için ızgaradaki her noktanın eğitim noktalarıyla çekirdeğini hesaplarız.

In [ ]:
def qsvm(Xtr, ytr, fmap="zz", gamma=1.0, C=1.0):
    model = SVC(kernel="precomputed", C=C).fit(qkernel(Xtr, fmap=fmap, gamma=gamma), ytr)
    return model

def qsvm_score(model, Xtr, X, y, fmap, gamma):
    return model.score(qkernel(X, Xtr, fmap=fmap, gamma=gamma), y)

Xtr, Xte, ytr, yte, _, _ = SPLIT["moons"]
for g in [1.0, 0.5]:
    mdl = qsvm(Xtr, ytr, "zz", g)
    print(f"moons, ZZ γ={g}: eğitim={qsvm_score(mdl, Xtr, Xtr, ytr, 'zz', g):.3f}  test={qsvm_score(mdl, Xtr, Xte, yte, 'zz', g):.3f}  SV={len(mdl.support_)}")

In [ ]:
# Gram ısı haritaları (ZZ, sınıfa göre sıralı)
GSEL = {"moons": 0.5, "circles": 0.5, "xor": 0.3, "iris2": 0.1}     # eğitim kümesinde iç 5-kat CV ile seçilen γ (make_images.py ile aynı)
fig, axs = plt.subplots(1, 4, figsize=(16, 3.8))
for ax, k in zip(axs, DS):
    Xtr_, _, ytr_, _, _, _ = SPLIT[k]; o = np.argsort(ytr_, kind="stable")
    ax.imshow(qkernel(Xtr_[o], fmap="zz", gamma=GSEL[k]), cmap=CMK, vmin=0, vmax=1)
    n0 = (ytr_ == 0).sum(); ax.axhline(n0 - .5, color=ORANGE); ax.axvline(n0 - .5, color=ORANGE)
    ax.set_title(f"{k} (γ={GSEL[k]})", color=NAVY); ax.set_xticks([]); ax.set_yticks([])
plt.tight_layout(); plt.show()

In [ ]:
# Karar sınırları: ızgara tahmini tamamen vektörleştirilmiş (70×70 = 4900 nokta)
gg = np.linspace(-0.1, np.pi + 0.1, 70); GX, GY = np.meshgrid(gg, gg); G = np.c_[GX.ravel(), GY.ravel()]

def decision_grid(k, kind, gamma=None):
    Xtr_, Xte_, ytr_, yte_, _, _ = SPLIT[k]
    if gamma is None:
        m = SVC(kernel=kind).fit(Xtr_, ytr_); return m.decision_function(G), m.score(Xte_, yte_)
    m = SVC(kernel="precomputed").fit(qkernel(Xtr_, fmap=kind, gamma=gamma), ytr_)
    return m.decision_function(qkernel(G, Xtr_, fmap=kind, gamma=gamma)), m.score(qkernel(Xte_, Xtr_, fmap=kind, gamma=gamma), yte_)

t = time.time()
fig, axs = plt.subplots(3, 4, figsize=(15, 10.5))
for r, k in enumerate(["moons", "circles", "xor"]):
    Xtr_, Xte_, ytr_, yte_, _, _ = SPLIT[k]
    for c, (tl, kind, g) in enumerate([("Doğrusal", "linear", None), ("RBF", "rbf", None), ("QSVM açı γ=1", "angle", 1.0), (f"QSVM ZZ γ={GSEL[k]}", "zz", GSEL[k])]):
        Z, te = decision_grid(k, kind, g); Z = Z.reshape(GX.shape); ax = axs[r, c]
        ax.contourf(GX, GY, Z > 0, cmap=CM2); ax.contour(GX, GY, Z, levels=[0], colors=NAVY)
        ax.scatter(*Xtr_.T, c=np.where(ytr_ == 0, BLUE, ORANGE), s=9)
        ax.scatter(*Xte_.T, c=np.where(yte_ == 0, BLUE, ORANGE), s=24, marker="^", edgecolor="k", lw=.4)
        ax.set_title(f"{k} · {tl}\ntest = {te:.2f}", fontsize=10, color=NAVY); ax.set_xticks([]); ax.set_yticks([])
plt.tight_layout(); plt.show()
print(f"12 karar sınırı {time.time()-t:.1f} s")

In [ ]:
# Karşılaştırma tablosu: eğitim / test doğruluğu + 5-kat çapraz doğrulama
CV = StratifiedKFold(5, shuffle=True, random_state=SEED)

def cv_qkernel(X, y, fmap, gamma):
    K = qkernel(X, fmap=fmap, gamma=gamma); s = []
    for tr, te in CV.split(X, y):
        m = SVC(kernel="precomputed").fit(K[np.ix_(tr, tr)], y[tr]); s.append(m.score(K[np.ix_(te, tr)], y[te]))
    return np.mean(s)

rows = []
for k in DS:
    Xtr_, Xte_, ytr_, yte_, Xa, ya = SPLIT[k]
    for nm, kind in [("Doğrusal SVM", "linear"), ("RBF SVM", "rbf")]:
        m = SVC(kernel=kind).fit(Xtr_, ytr_)
        rows.append([k, nm, m.score(Xtr_, ytr_), m.score(Xte_, yte_), cross_val_score(SVC(kernel=kind), Xa, ya, cv=CV).mean()])
    for nm, fmap, g in [("QSVM açı (γ=1)", "angle", 1.0), ("QSVM ZZ (γ=1)", "zz", 1.0), ("QSVM ZZ (γ ayarlı)", "zz", GSEL[k]), ("QSVM özel (γ=1)", "custom", 1.0)]:
        m = qsvm(Xtr_, ytr_, fmap, g)
        rows.append([k, nm, qsvm_score(m, Xtr_, Xtr_, ytr_, fmap, g), qsvm_score(m, Xtr_, Xte_, yte_, fmap, g), cv_qkernel(Xa, ya, fmap, g)])
table = pd.DataFrame(rows, columns=["veri", "model", "eğitim", "test", "5-kat CV"])
table.pivot(index="model", columns="veri", values="5-kat CV").round(3)

In [ ]:
table.round(3)

**Yorum (dürüst değerlendirme):** Bu küçük, 2–4 boyutlu veri setlerinde **RBF SVM zaten ≈ %100**'e ulaşıyor. Uygun γ ile QSVM onu ancak **yakalıyor**, geçemiyor. Varsayılan γ = 1 ile ZZ haritası belirgin şekilde **daha kötü**. Kuantum çekirdek sihirli değildir: iyi bir özellik haritası + iyi ayarlanmış bant genişliği gerekir.

---
## F · Bant genişliği (γ) taraması
γ küçük → tüm durumlar |0…0⟩'a yakın → K ≈ hep 1 (**her şey aynı**). γ büyük → durumlar çok hızlı değişir → K ≈ birim matris (**her şey farklı**, ezberleme). Arada bir tatlı nokta vardır.

In [ ]:
GAMMAS = [0.1, 0.2, 0.3, 0.5, 0.7, 1.0, 1.5, 2.0]
t = time.time()
fig, axs = plt.subplots(1, 4, figsize=(16, 3.6), sharey=True)
sweep = {}
for ax, k in zip(axs, DS):
    Xa, ya = SPLIT[k][4], SPLIT[k][5]
    for fmap, c in [("angle", "#9DBBE3"), ("zz", BLUE), ("custom", NAVY)]:
        sweep[(k, fmap)] = [cv_qkernel(Xa, ya, fmap, g) for g in GAMMAS]
        ax.plot(GAMMAS, sweep[(k, fmap)], "o-", color=c, label=fmap)
    ax.axhline(cross_val_score(SVC(kernel="rbf"), Xa, ya, cv=CV).mean(), color=ORANGE, ls="--", label="RBF")
    ax.set_xscale("log"); ax.set_title(k, color=NAVY); ax.set_xlabel("γ"); ax.grid(alpha=.25)
axs[0].set_ylabel("5-kat CV doğruluğu"); axs[0].legend(frameon=False, fontsize=9)
plt.tight_layout(); plt.show()
print(f"tarama {time.time()-t:.1f} s")
for k in DS:
    i = int(np.argmax(sweep[(k, 'zz')])); print(f"{k:8s} en iyi ZZ γ = {GAMMAS[i]}  CV = {sweep[(k, 'zz')][i]:.3f}")

In [ ]:
# γ'nın Gram matrisine etkisi (moons, 40 örnek)
Xtr, _, ytr, _, _, _ = SPLIT["moons"]; sel = np.r_[np.where(ytr == 0)[0][:20], np.where(ytr == 1)[0][:20]]
fig, axs = plt.subplots(1, 3, figsize=(13, 4))
for ax, g in zip(axs, [0.05, 0.5, 2.0]):
    K = qkernel(Xtr[sel], fmap="zz", gamma=g); off = K[np.triu_indices(40, 1)]
    ax.imshow(K, cmap=CMK, vmin=0, vmax=1); ax.set_title(f"γ={g}: ort={off.mean():.2f}, std={off.std():.2f}", color=NAVY); ax.set_xticks([]); ax.set_yticks([])
plt.tight_layout(); plt.show()

---
## G · Shot tabanlı QSVM ve maliyet
Gerçek donanımda her K[i, j] bir compute–uncompute devresi ve S shot ile **tahmin edilir**. Tahminin standart hatası ≈ √(k(1 − k)/S). Önce küçük bir alt kümede (20 eğitim, 20 test) gerçekten **Aer ile devre çalıştırarak** QSVM kuralım.

In [ ]:
Xtr, Xte, ytr, yte, _, _ = SPLIT["moons"]
tr_idx = np.r_[np.where(ytr == 0)[0][:10], np.where(ytr == 1)[0][:10]]
te_idx = np.r_[np.where(yte == 0)[0][:10], np.where(yte == 1)[0][:10]]
Xs, ys, Xq, yq = Xtr[tr_idx], ytr[tr_idx], Xte[te_idx], yte[te_idx]
GAMMA, SHOTS = 0.5, 256
fm2 = zz_feature_map(2, reps=2)
zz_fn = lambda x: fm2.assign_parameters(GAMMA*np.asarray(x))

t = time.time()
pairs_tr = [(i, j) for i in range(len(Xs)) for j in range(i + 1, len(Xs))]       # simetri: yalnız üst üçgen
pairs_te = [(i, j) for i in range(len(Xq)) for j in range(len(Xs))]
circs = [cu_circuit(zz_fn, Xs[i], Xs[j]) for i, j in pairs_tr] + [cu_circuit(zz_fn, Xq[i], Xs[j]) for i, j in pairs_te]
aer = AerSimulator(seed_simulator=SEED)
res = aer.run(transpile(circs, aer, optimization_level=0), shots=SHOTS).result()
p00 = np.array([res.get_counts(i).get("00", 0)/SHOTS for i in range(len(circs))])
print(f"{len(circs)} devre × {SHOTS} shot = {len(circs)*SHOTS:,} shot, süre {time.time()-t:.1f} s")

K_tr_shot = np.eye(len(Xs))
for (i, j), v in zip(pairs_tr, p00[:len(pairs_tr)]): K_tr_shot[i, j] = K_tr_shot[j, i] = v
K_te_shot = p00[len(pairs_tr):].reshape(len(Xq), len(Xs))
K_tr_ex, K_te_ex = qkernel(Xs, fmap="zz", gamma=GAMMA), qkernel(Xq, Xs, fmap="zz", gamma=GAMMA)
print("ortalama |K_shot − K_kesin| =", np.abs(K_tr_shot - K_tr_ex).mean().round(4))
acc_shot = SVC(kernel="precomputed").fit(K_tr_shot, ys).score(K_te_shot, yq)
acc_ex = SVC(kernel="precomputed").fit(K_tr_ex, ys).score(K_te_ex, yq)
print(f"test doğruluğu: shot tabanlı = {acc_shot:.2f}, kesin (Statevector) = {acc_ex:.2f}")

In [ ]:
# Shot sayısının etkisi: shot gürültüsünü Binom(S, k)/S ile taklit ederek (hızlı)
fig, axs = plt.subplots(1, 3, figsize=(14, 3.8))
for ax, S in zip(axs, [16, 256, None]):
    Kx = K_tr_ex if S is None else shot_kernel(K_tr_ex, S, seed=1)
    ax.imshow(Kx, cmap=CMK, vmin=0, vmax=1); ax.set_title("kesin" if S is None else f"S = {S} shot", color=NAVY); ax.set_xticks([]); ax.set_yticks([])
plt.tight_layout(); plt.show()
for S in [8, 32, 128, 1024]:
    accs = [SVC(kernel="precomputed").fit(shot_kernel(K_tr_ex, S, seed=r), ys).score(shot_kernel(K_te_ex, S, seed=100 + r), yq) for r in range(10)]
    print(f"S = {S:5d}: test doğruluğu ort = {np.mean(accs):.2f} ± {np.std(accs):.2f}")

### Maliyet hesabı
| Aşama | Devre sayısı | Not |
|---|---|---|
| Eğitim Gram | N(N − 1)/2 | köşegen = 1, simetri |
| Test | M × N_SV | yalnız destek vektörleri gerekir |
| Toplam shot | devre × S | S: eleman başına shot |

In [ ]:
def cost(N, M, S, n_sv=None):
    tr = N*(N - 1)//2; te = M*(n_sv if n_sv else N)
    return tr, te, (tr + te)*S

for N in [20, 140, 1000]:
    tr, te, tot = cost(N, int(0.3*N/0.7), 1000)
    print(f"N={N:5d}: eğitim devresi={tr:>9,}  test devresi={te:>9,}  toplam shot={tot:>14,}")

---
## H · Çekirdek yoğunlaşması (exponential concentration)
Kübit sayısı arttıkça rastgele iki verinin durumları neredeyse **dik** olur: k(x, x') değerleri 0'a yığılır, varyansları **üstel** azalır. Sonuç: Gram matrisi ≈ birim matris, SVM ezberler; ayrıca 0'a çok yakın değerleri shot ile ayırt etmek için üstel sayıda shot gerekir. **Çözüm:** bant genişliğini kübit sayısıyla küçültmek (ör. ZZ için γ ∝ 1/n²).

In [ ]:
rng = np.random.default_rng(7); conc = []
t = time.time()
for n in range(2, 11):
    Xr = rng.uniform(0, np.pi, (60, n))
    row = [n]
    for g in [1.0, 1.0/n**2]:
        off = qkernel(Xr, fmap="zz", gamma=g)[np.triu_indices(60, 1)]
        row += [off.mean(), off.var()]
    conc.append(row)
conc = pd.DataFrame(conc, columns=["n", "ort (γ=1)", "var (γ=1)", "ort (γ=1/n²)", "var (γ=1/n²)"])
fig, ax = plt.subplots(figsize=(7, 3.6))
ax.semilogy(conc.n, conc["var (γ=1)"], "o-", color=ORANGE, label="γ = 1"); ax.semilogy(conc.n, conc["var (γ=1/n²)"], "o-", color=BLUE, label="γ = 1/n²")
ax.set_xlabel("kübit sayısı n"); ax.set_ylabel("Var[k]"); ax.legend(frameon=False); ax.grid(alpha=.25); plt.show()
print(f"süre {time.time()-t:.1f} s"); conc.round(4)

---
## I · PennyLane ile aynı çekirdek (genel kültür)
PennyLane'de compute–uncompute: `AngleEmbedding` + `qml.adjoint(AngleEmbedding)` ve `qml.probs` ile |0…0⟩ olasılığı. `qml.kernels.square_kernel_matrix` Gram matrisini kurar.

In [ ]:
import pennylane as qml

n_q = 2
dev = qml.device("default.qubit", wires=n_q)

@qml.qnode(dev)
def overlap(x1, x2):
    qml.AngleEmbedding(x1, wires=range(n_q), rotation="Y")              # U(x)
    qml.adjoint(qml.AngleEmbedding)(x2, wires=range(n_q), rotation="Y") # U(x')†
    return qml.probs(wires=range(n_q))

pl_kernel = lambda a, b: overlap(a, b)[0]                              # P(00)
Xs5 = SPLIT["moons"][0][:5]
K_pl = qml.kernels.square_kernel_matrix(Xs5, pl_kernel, assume_normalized_kernel=True)
print("PennyLane == bizim açı çekirdeğimiz ?", np.allclose(K_pl, qkernel(Xs5, fmap="angle")))

---
## J · Alıştırmalar
`# TODO` yerlerini doldurun; `assert` satırları geçerse çözüm doğrudur. Tüm sonuçlar sabit seed ile deterministiktir.

### Alıştırma 1 · Açı çekirdeği (kapalı formül)
`angle_kernel(x, xp, gamma=1.0)` fonksiyonu k = Π cos²(γ(xᵢ − x'ᵢ)/2) döndürsün.

In [ ]:
def angle_kernel(x, xp, gamma=1.0):
    # TODO
    pass

assert np.isclose(angle_kernel([0.5], [1.5]), np.cos(0.5)**2)
assert np.isclose(angle_kernel([0.3, 1.0], [1.1, 0.4]), 0.7743, atol=1e-3)
assert np.isclose(angle_kernel([1, 2, 3], [1, 2, 3]), 1.0)
assert np.isclose(angle_kernel([0.2, 2.0], [1.4, 0.6], gamma=0.5), qkernel(np.array([[0.2, 2.0]]), np.array([[1.4, 0.6]]), "angle", 0.5)[0, 0])
print("Alıştırma 1 ✓")

### Alıştırma 2 · Durum matrisinden Gram matrisi
`gram_from_states(S)`: S'nin her satırı bir durum vektörü. K[i, j] = |⟨Sᵢ|Sⱼ⟩|² matrisini **döngüsüz** (tek matris çarpımı) hesaplayın.

In [ ]:
def gram_from_states(S):
    # TODO
    pass

S = zz_states(SPLIT["xor"][0][:30], gamma=0.3)
K = gram_from_states(S)
assert K.shape == (30, 30) and np.allclose(K, K.T) and np.allclose(np.diag(K), 1)
assert np.linalg.eigvalsh(K).min() > -1e-9          # pozitif yarı-tanımlı
assert np.allclose(K, qkernel(SPLIT["xor"][0][:30], fmap="zz", gamma=0.3))
print("Alıştırma 2 ✓")

### Alıştırma 3 · Sayımlardan çekirdek değeri
`kernel_from_counts(counts)`: compute–uncompute devresinin sayımlarından k ≈ P(tümü sıfır) tahmini döndürsün (kübit sayısını anahtarın uzunluğundan bulun).

In [ ]:
def kernel_from_counts(counts):
    # TODO
    pass

assert np.isclose(kernel_from_counts({"00": 639, "01": 209, "10": 131, "11": 45}), 639/1024)
assert np.isclose(kernel_from_counts({"101": 10, "000": 30}), 0.75)
assert kernel_from_counts({"11": 5}) == 0.0
qc = cu_circuit(angle_circuit, [0.3, 1.0], [1.1, 0.4])
cnt = AerSimulator(seed_simulator=3).run(transpile(qc, AerSimulator()), shots=4000).result().get_counts()
assert abs(kernel_from_counts(cnt) - 0.7743) < 0.03
print("Alıştırma 3 ✓", kernel_from_counts(cnt))

### Alıştırma 4 · Angle compute–uncompute devresi
`cu_angle(x, xp)`: açı kodlaması için compute–uncompute devresini **ölçümsüz** kurun (RY(xᵢ) sonra RY(−x'ᵢ)). Statevector ile |0…0⟩ genliğinin karesinin kapalı formüle eşit olduğunu doğrulayın.

In [ ]:
def cu_angle(x, xp):
    # TODO
    pass

for x, xp in [([0.3, 1.0], [1.1, 0.4]), ([2.0, 0.1, 1.5], [0.5, 0.9, 1.5])]:
    p0 = abs(Statevector(cu_angle(x, xp)).data[0])**2
    assert np.isclose(p0, angle_kernel(x, xp))
assert cu_angle([0.1, 0.2], [0.3, 0.4]).num_qubits == 2
print("Alıştırma 4 ✓")

### Alıştırma 5 · QSVM test doğruluğu
`qsvm_test_acc(name, fmap, gamma)`: `SPLIT[name]` üzerinde precomputed SVC ile QSVM eğitip **test doğruluğunu** döndürsün. Circles için ZZ (γ = 0.5) ve XOR için açı (γ = 1) çekirdeğiyle eşiklere ulaşın.

In [ ]:
def qsvm_test_acc(name, fmap, gamma):
    Xtr, Xte, ytr, yte, _, _ = SPLIT[name]
    # TODO
    pass

a1, a2 = qsvm_test_acc("circles", "zz", 0.5), qsvm_test_acc("xor", "angle", 1.0)
print(a1, a2)
assert a1 >= 0.95 and a2 >= 0.95
assert qsvm_test_acc("moons", "zz", 2.0) < 0.7        # γ çok büyük: ezber
print("Alıştırma 5 ✓")

### Alıştırma 6 · Çapraz doğrulama ile γ seçimi
`best_gamma(name, fmap, grid)`: tüm veri üzerinde 5-kat CV (`CV` nesnesi) ile en iyi γ'yı ve skorunu döndürsün (`cv_qkernel` kullanabilirsiniz). Moons + ZZ için sonucu RBF'in CV skoruyla karşılaştırın.

In [ ]:
def best_gamma(name, fmap, grid):
    Xa, ya = SPLIT[name][4], SPLIT[name][5]
    # TODO: (en iyi γ, en iyi skor)
    pass

g, s = best_gamma("moons", "zz", [0.1, 0.3, 0.5, 1.0, 2.0])
rbf = cross_val_score(SVC(kernel="rbf"), SPLIT["moons"][4], SPLIT["moons"][5], cv=CV).mean()
print(f"en iyi γ = {g}, CV = {s:.3f}, RBF CV = {rbf:.3f}")
assert g == 0.5 and s >= 0.95
assert s <= rbf + 1e-9              # dürüst sonuç: RBF'i geçemedi
print("Alıştırma 6 ✓")

### Alıştırma 7 · Maliyet hesabı
`kernel_cost(N, M, S, n_sv=None)`: (eğitim devre sayısı, test devre sayısı, toplam shot) döndürsün. Eğitimde simetri ve köşegen = 1 kullanılır; testte yalnız destek vektörleri (verilmişse) gerekir.

In [ ]:
def kernel_cost(N, M, S, n_sv=None):
    # TODO
    pass

assert kernel_cost(140, 60, 1000) == (9730, 8400, 18_130_000)
assert kernel_cost(140, 60, 1000, n_sv=58) == (9730, 3480, 13_210_000)
assert kernel_cost(1000, 0, 1)[0] == 499_500
print("Alıştırma 7 ✓")

### Alıştırma 8 · Yoğunlaşmayı gözlemle ve düzelt
`offdiag_mean(n, gamma)`: `rng = np.random.default_rng(7)` ile [0, π]'de 50 rastgele n boyutlu nokta üretip ZZ çekirdeğinin köşegen dışı ortalamasını döndürsün. n = 8 için γ = 1 ve γ = 1/n² değerlerini karşılaştırın.

In [ ]:
def offdiag_mean(n, gamma):
    # TODO
    pass

m1, m2 = offdiag_mean(8, 1.0), offdiag_mean(8, 1/64)
print(f"n=8: γ=1 -> {m1:.4f}   γ=1/n² -> {m2:.4f}")
assert m1 < 0.02                    # değerler 0'a yığıldı
assert m2 > 0.4                     # küçük γ benzerlik bilgisini korudu
assert offdiag_mean(2, 1.0) > offdiag_mean(6, 1.0)
print("Alıştırma 8 ✓")

In [ ]:
print(f"Toplam çalışma süresi: {time.time() - T_START:.0f} s")

---
### Haftanın özeti
- Kuantum çekirdek: **k(x, x') = |⟨φ(x)|φ(x')⟩|²**; özellik haritası = veri kodlama devresi U(x)
- Hesap yolları: Statevector (kesin, simülatör), **compute–uncompute** (P(00…0), donanım), swap test (2n+1 kübit)
- QSVM = kuantum Gram matrisi + klasik `SVC(kernel="precomputed")`; test için K(test, eğitim)
- **Bant genişliği γ** kritik: çok küçük → her şey aynı, çok büyük → K ≈ I (ezber)
- Maliyet N² devre × S shot; shot hatası ≈ 1/√S
- **Yoğunlaşma:** kübit arttıkça k → 0; çözüm γ'yı küçültmek
- Bu veri setlerinde **RBF SVM eşit veya daha iyi**; kuantum avantajı ancak özel yapılı verilerde iddia edilir

**Gelecek hafta (Hafta 14):** Varyasyonel sınıflandırıcılar (VQC) ve hibrit kuantum sinir ağları: eğitilebilir devreler, PennyLane + PyTorch, barren plateau.